In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,roc_auc_score,average_precision_score

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
import joblib

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
DATA_PATH = "/content/drive/MyDrive/ColabNotebooks/Santander_Competition/"

In [6]:
df_train=pd.read_csv(f'{DATA_PATH}/train.csv')

In [7]:
x=df_train.drop(['ID_code','target'],axis=1)
y=df_train['target']

**tuning parameters**

In [8]:
# Small stratified sample for hyperparameter tuning
x_tune, _, y_tune, _ = train_test_split( x, y, train_size=0.10, random_state=42, stratify=y)

**Random Forest balanced**

In [9]:
model_RF=RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced', n_jobs=1)

**buscar mejores hiperparametros**

In [10]:
cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

param_grid_rf = {
    'max_depth': [5,8,10, 15],
    'min_samples_leaf': [2,5,10,20],
    'max_features': ['sqrt', 0.3, 0.5],
    'min_samples_split': [2,5,10]
            }

In [ ]:
random_cv_RF = RandomizedSearchCV(estimator = model_RF,
                               param_distributions = param_grid_rf,
                               random_state=42,
                               cv = cv_strategy,
                               n_iter = 25,
                               scoring = 'roc_auc',
                               verbose = 5,
                               return_train_score = True,
                               n_jobs=1 )
random_cv_RF.fit(x_tune, y_tune)

random_cv_RF.best_estimator_
print("Best ROC-AUC CV:", random_cv_RF.best_score_)
print("Best parameters:")
print(random_cv_RF.best_params_)


Fitting 3 folds for each of 25 candidates, totalling 75 fits
[CV 1/3] END max_depth=15, max_features=sqrt, min_samples_leaf=20, min_samples_split=2;, score=(train=1.000, test=0.822) total time= 1.5min
[CV 2/3] END max_depth=15, max_features=sqrt, min_samples_leaf=20, min_samples_split=2;, score=(train=1.000, test=0.842) total time= 1.5min
[CV 3/3] END max_depth=15, max_features=sqrt, min_samples_leaf=20, min_samples_split=2;, score=(train=1.000, test=0.846) total time= 1.4min
[CV 1/3] END max_depth=5, max_features=0.3, min_samples_leaf=10, min_samples_split=5;, score=(train=0.891, test=0.726) total time= 2.8min
[CV 2/3] END max_depth=5, max_features=0.3, min_samples_leaf=10, min_samples_split=5;, score=(train=0.907, test=0.753) total time= 2.8min
[CV 3/3] END max_depth=5, max_features=0.3, min_samples_leaf=10, min_samples_split=5;, score=(train=0.903, test=0.759) total time= 2.8min
[CV 1/3] END max_depth=10, max_features=sqrt, min_samples_leaf=20, min_samples_split=5;, score=(train=1.0

**running with the best parameter**

In [8]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42, stratify=y)

In [9]:
#final_params = random_cv_RF.best_params_

#model_RF_tuned=RandomForestClassifier(**final_params, random_state=42, class_weight='balanced',n_estimators=600)

model_RF_tuned=RandomForestClassifier(max_depth=15, max_features='sqrt',min_samples_leaf=10, min_samples_split=5, n_estimators=600, random_state=42, class_weight='balanced', n_jobs=1)
model_RF_tuned.fit(x_train,y_train)

RandomForestClassifier(class_weight='balanced', max_depth=15,
                       min_samples_leaf=10, min_samples_split=5,
                       n_estimators=600, n_jobs=1, random_state=42)

In [10]:
y_pred=model_RF_tuned.predict(x_test)
y_prob = model_RF_tuned.predict_proba(x_test)[:,1]

print('accuracy: ' + str(accuracy_score(y_test,y_pred)))
print('confusion matrix: \n' + str(confusion_matrix(y_test,y_pred)))
print('classification report: \n' + str(classification_report(y_test,y_pred)))
print('roc_auc_score: ' + str(roc_auc_score(y_test,y_prob)))
print('pr_auc_score: ' + str(average_precision_score(y_test,y_prob)))

accuracy: 0.903075
confusion matrix: 
[[35941    39]
 [ 3838   182]]
classification report: 
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     35980
           1       0.82      0.05      0.09      4020

    accuracy                           0.90     40000
   macro avg       0.86      0.52      0.52     40000
weighted avg       0.90      0.90      0.86     40000

roc_auc_score: 0.8321001233410491
pr_auc_score: 0.4240177031398116


In [11]:
for t in [0.2, 0.3, 0.4, 0.5]:
    y_pred_t = (y_prob > t).astype(int)
    print(f"\nThreshold: {t}")
    print(confusion_matrix(y_test, y_pred_t))
    print(classification_report(y_test, y_pred_t))


Threshold: 0.2
[[    0 35980]
 [    0  4020]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     35980
           1       0.10      1.00      0.18      4020

    accuracy                           0.10     40000
   macro avg       0.05      0.50      0.09     40000
weighted avg       0.01      0.10      0.02     40000


Threshold: 0.3
[[ 8106 27874]
 [   66  3954]]
              precision    recall  f1-score   support

           0       0.99      0.23      0.37     35980
           1       0.12      0.98      0.22      4020

    accuracy                           0.30     40000
   macro avg       0.56      0.60      0.29     40000
weighted avg       0.90      0.30      0.35     40000


Threshold: 0.4
[[32063  3917]
 [ 1809  2211]]
              precision    recall  f1-score   support

           0       0.95      0.89      0.92     35980
           1       0.36      0.55      0.44      4020

    accuracy                           0.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
rf_results = pd.DataFrame({
    'y_true': y_test.values,
    'rf_probability': y_prob,
    'rf_prediction_05': y_pred
})

rf_results.to_csv(f'{DATA_PATH}/rf_validation_results.csv', index=False)

best_params = model_RF_tuned.get_params()

joblib.dump(model_RF_tuned, f'{DATA_PATH}/best_random_forest_model.pkl')
joblib.dump(best_params, f'{DATA_PATH}/best_random_forest_params.pkl')

['/content/drive/MyDrive/ColabNotebooks/Santander_Competition//best_random_forest_params.pkl']